# Learning to See in the Dark

**Paper**: *Learning to See in the Dark*, Chen Chen, Qifeng Chen, Jia Xu, Vladlen Koltun — CVPR 2018

## Key Contributions
1. **SID Dataset** — 5,094 raw short-exposure low-light image pairs (Sony α7SII + Fujifilm X-T2).
2. **End-to-End Pipeline** — A fully-convolutional network that operates *directly on raw sensor data*, replacing the entire traditional ISP (white balance, demosaicing, denoising, colour conversion, gamma).
3. **Architecture** — U-Net with a sub-pixel (PixelShuffle) output layer. Input: packed 4-channel Bayer (H/2 × W/2). Output: 12-channel → PixelShuffle(2) → full-resolution RGB.
4. **Training** — L1 loss, Adam optimiser, 4,000 epochs, 512 × 512 random crops, random flip/rotation augmentation.

## Notebook Overview
| Section | Description |
|---------|-------------|
| 1 | Environment setup & GPU check |
| 2 | Dataset download & file-list parsing |
| 3 | Raw image preprocessing (Bayer packing, black-level, amplification) |
| 4 | Custom PyTorch `Dataset` and `DataLoader` |
| 5 | U-Net model with PixelShuffle output |
| 6 | Training loop with AMP mixed-precision & checkpointing |
| 7 | Evaluation — PSNR / SSIM |
| 8 | Inference & visualisation |

> **Scaling for speed** — images are scaled down (`SCALE_FACTOR`) before training crops are taken, and patch size can be reduced independently. Default settings run comfortably on a free Colab T4.

---
## Section 1 — Environment Setup

In [ ]:
# Install required packages
!pip install -q rawpy imageio scikit-image tqdm gdown

In [ ]:
import subprocess, sys, os, glob, random, time, math
from pathlib import Path

import numpy as np
import rawpy
import imageio
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast

# ── GPU check ────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM: {total_mem:.1f} GB')
else:
    print('  WARNING: No GPU found. Training will be very slow.')
    print('  Go to Runtime → Change runtime type → GPU')

---
## Section 2 — Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CONFIGURATION  —  adjust these to trade speed for quality
# ═══════════════════════════════════════════════════════════════

# Paths
BASE_DIR   = Path('/content/SID')          # root of downloaded dataset
SONY_DIR   = BASE_DIR / 'Sony'
CKPT_DIR   = Path('/content/checkpoints')
RESULT_DIR = Path('/content/results')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# ── Speed / quality trade-offs ───────────────────────────────
# SCALE_FACTOR < 1 downsamples raw images before packing → faster I/O
# and smaller tensors.  0.5 = half resolution.  1.0 = full resolution.
SCALE_FACTOR = 0.5          # 0.25 | 0.5 | 1.0

# PATCH_SIZE is the crop size taken from the *original* raw image
# (before packing).  Packed size = PATCH_SIZE // 2.
# Paper: 512.  For speed on free Colab: 256 or 512.
PATCH_SIZE  = 512

# Training hyper-parameters
BATCH_SIZE  = 4             # increase if VRAM allows
NUM_EPOCHS  = 4000          # set lower (e.g. 500) for a quick demo
LR_INITIAL  = 1e-4
LR_DECAY_EP = 2000          # halve LR at this epoch
LR_DECAYED  = 1e-5
NUM_WORKERS = 2             # DataLoader workers

# Misc
SEED        = 42
SAVE_EVERY  = 200           # save checkpoint every N epochs
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'SCALE_FACTOR = {SCALE_FACTOR}')
print(f'PATCH_SIZE   = {PATCH_SIZE}')
print(f'BATCH_SIZE   = {BATCH_SIZE}')
print(f'NUM_EPOCHS   = {NUM_EPOCHS}')

---
## Section 3 — Dataset Download

The **See-in-the-Dark (SID)** dataset is the official benchmark introduced by this paper.

### Download options (choose one)

**Option A — Google Drive mount (recommended for large dataset)**
```
from google.colab import drive
drive.mount('/content/drive')
# Then copy / symlink from your Drive folder to /content/SID
```

**Option B — Direct download via gdown** (run the cell below)
- Sony subset ≈ 25 GB; downloading takes ~20 min on Colab.
- File-list `.txt` files are fetched automatically from the official GitHub repo.

**Option C — Small demo subset** — Run the *synthetic demo* cell at the bottom of this section to train on tiny synthetic data without downloading anything.

In [ ]:
# ── Option B: Download Sony subset from the official Google Drive ─
# Uncomment and run this cell to download the full Sony subset.

import os

DOWNLOAD_DATASET = False   # ← Set to True to trigger download

if DOWNLOAD_DATASET:
    os.makedirs('/content/SID', exist_ok=True)

    # Sony subset (≈25 GB ARW raw files)
    # Official Google Drive ID from: https://github.com/cchen156/Learning-to-See-in-the-Dark
    SONY_GDRIVE_ID = '10kpAcvldtcb9G2ze5hTcF1odzu4V_Zvh'
    print('Downloading Sony.zip (≈25 GB) …')
    !gdown --id {SONY_GDRIVE_ID} -O /content/SID/Sony.zip
    print('Extracting …')
    !unzip -q /content/SID/Sony.zip -d /content/SID/
    print('Done.')

# ── Download train / val / test file-lists from GitHub ───────────
LISTS_BASE = 'https://raw.githubusercontent.com/cchen156/Learning-to-See-in-the-Dark/master'
for split in ['train', 'val', 'test']:
    fname = f'Sony_{split}_list.txt'
    dest  = BASE_DIR / fname
    if not dest.exists():
        BASE_DIR.mkdir(parents=True, exist_ok=True)
        !wget -q -O {str(dest)} {LISTS_BASE}/{fname}
        print(f'Downloaded {fname}')
    else:
        print(f'{fname} already present.')

In [ ]:
# ── Option C: Synthetic demo (no download needed) ────────────────
# Creates tiny fake ARW-like data so every subsequent cell works.
# Skip this cell if you have the real dataset.

USE_SYNTHETIC_DEMO = True   # ← Set to False when real dataset is ready

if USE_SYNTHETIC_DEMO:
    import warnings
    warnings.warn(
        'Running in SYNTHETIC DEMO mode. '
        'Results will not be meaningful — this mode only verifies the pipeline.'
    )

    # We will generate random numpy arrays that mimic packed Bayer inputs
    # and sRGB ground truths, bypassing rawpy entirely.
    DEMO_N_TRAIN = 20
    DEMO_N_VAL   = 4
    DEMO_N_TEST  = 4
    DEMO_H       = 256   # raw height (before packing → 128)
    DEMO_W       = 256   # raw width  (before packing → 128)
    print(f'Synthetic demo: {DEMO_N_TRAIN} train, {DEMO_N_VAL} val, {DEMO_N_TEST} test samples')
    print(f'Synthetic image size: {DEMO_H}×{DEMO_W} (raw) → {DEMO_H//2}×{DEMO_W//2}×4 (packed)')
else:
    USE_SYNTHETIC_DEMO = False
    print('Real dataset mode — ensure /content/SID/Sony is populated.')

---
## Section 4 — Raw Image Preprocessing

### Bayer packing (Figure 3b in the paper)
The Sony camera produces a single-channel 14-bit Bayer image in the RGGB pattern:
```
R  Gr
Gb  B
```
We pack it into **4 channels** and simultaneously halve the spatial resolution:
```
channel 0 : R  [rows 0::2, cols 0::2]
channel 1 : Gr [rows 0::2, cols 1::2]
channel 2 : B  [rows 1::2, cols 1::2]
channel 3 : Gb [rows 1::2, cols 0::2]
```
After packing an H × W raw image → (H/2) × (W/2) × 4 tensor.

### Amplification
Each input image is multiplied by the ratio
```
amp = exposure_time_GT / exposure_time_input   (typically 100 – 300)
```
before being fed to the network (analogous to the ISO knob on a camera).

In [ ]:
# ────────────────────────────────────────────────────────────────
#  Raw-image utilities
# ────────────────────────────────────────────────────────────────

def extract_exposure(filename: str) -> float:
    """Parse exposure time (seconds) from SID filename.
    Examples: '00001_00_0.1s.ARW' → 0.1
              '00001_00_10s.ARW'  → 10.0
    """
    stem = Path(filename).stem           # e.g. '00001_00_0.1s'
    # last token before 's' at end
    t = stem.split('_')[-1]             # e.g. '0.1s'
    return float(t.rstrip('s'))


def pack_raw_bayer(raw_image: np.ndarray,
                   black_level: int = 512,
                   white_level: int = 16383) -> np.ndarray:
    """Pack a single-channel 14-bit Bayer (H×W) array into
    a 4-channel float32 array (H/2 × W/2 × 4) normalised to [0, 1].

    Channel order: R, Gr, B, Gb  (RGGB Bayer pattern).
    """
    im = raw_image.astype(np.float32)
    # Black-level subtraction and normalisation
    im = np.maximum(im - black_level, 0) / (white_level - black_level)

    H, W = im.shape
    # Ensure even dimensions
    H = H - H % 2
    W = W - W % 2
    im = im[:H, :W]

    out = np.stack([
        im[0::2, 0::2],   # R
        im[0::2, 1::2],   # Gr
        im[1::2, 1::2],   # B
        im[1::2, 0::2],   # Gb
    ], axis=-1)            # → (H/2, W/2, 4)
    return out


def load_input_raw(path: str,
                   amp_ratio: float,
                   scale: float = 1.0) -> np.ndarray:
    """Load a short-exposure ARW file and return a packed, amplified
    float32 array of shape (H', W', 4) clipped to [0, 1].

    scale  — optional spatial downscale factor applied *before* packing
             (uses nearest-neighbour on the raw sensor data to preserve
             Bayer mosaic alignment).
    """
    with rawpy.imread(path) as raw:
        bayer = raw.raw_image_visible.copy()
        black = int(raw.black_level_per_channel[0])  # use first channel's black level
        white = int(raw.white_level)

    if scale != 1.0:
        H, W = bayer.shape
        # Must preserve Bayer alignment: down-sample by integer factor of 2
        new_H = max(2, int(round(H * scale)))
        new_W = max(2, int(round(W * scale)))
        # Align to multiple of 2
        new_H = new_H - new_H % 2
        new_W = new_W - new_W % 2
        # Simple nearest-neighbour by slicing (preserves Bayer pattern)
        step_H = max(1, H // new_H)
        step_W = max(1, W // new_W)
        bayer = bayer[:new_H * step_H:step_H, :new_W * step_W:step_W]

    packed = pack_raw_bayer(bayer, black_level=black, white_level=white)
    packed = np.clip(packed * amp_ratio, 0.0, 1.0)
    return packed


def load_gt_rgb(path: str, scale: float = 1.0) -> np.ndarray:
    """Load a long-exposure ARW file, apply camera white balance,
    and return a float32 sRGB array of shape (H, W, 3) in [0, 1].

    scale  — optional spatial downscale applied after demosaicing.
    """
    with rawpy.imread(path) as raw:
        rgb = raw.postprocess(
            use_camera_wb=True,
            half_size=False,
            no_auto_bright=True,
            output_bps=16,
        )  # uint16, range [0, 65535]

    gt = rgb.astype(np.float32) / 65535.0

    if scale != 1.0:
        from skimage.transform import resize
        H, W = gt.shape[:2]
        new_H = max(1, int(round(H * scale)))
        new_W = max(1, int(round(W * scale)))
        gt = resize(gt, (new_H, new_W), anti_aliasing=True, preserve_range=True)
        gt = gt.astype(np.float32)

    return gt


print('Preprocessing utilities defined.')

---
## Section 5 — Dataset & DataLoader

In [ ]:
# ────────────────────────────────────────────────────────────────
#  File-list parsing helpers
# ────────────────────────────────────────────────────────────────

def parse_sony_list(txt_path: str, sony_root: str) -> list:
    """Parse a SID Sony_*_list.txt file.

    Returns a list of dicts with keys:
        'input_path', 'gt_path', 'amp_ratio'
    """
    pairs = []
    sony_root = Path(sony_root)
    with open(txt_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            in_rel, gt_rel = parts[0], parts[1]
            # Strip leading './' if present
            in_rel = in_rel.lstrip('./')
            gt_rel = gt_rel.lstrip('./')
            in_path = sony_root / in_rel
            gt_path = sony_root / gt_rel
            # Only include pairs where both files exist
            if not in_path.exists() or not gt_path.exists():
                continue
            t_in = extract_exposure(str(in_path))
            t_gt = extract_exposure(str(gt_path))
            amp  = t_gt / t_in if t_in > 0 else 300.0
            pairs.append({
                'input_path': str(in_path),
                'gt_path':    str(gt_path),
                'amp_ratio':  amp,
            })
    return pairs


if not USE_SYNTHETIC_DEMO:
    train_pairs = parse_sony_list(BASE_DIR / 'Sony_train_list.txt', SONY_DIR)
    val_pairs   = parse_sony_list(BASE_DIR / 'Sony_val_list.txt',   SONY_DIR)
    test_pairs  = parse_sony_list(BASE_DIR / 'Sony_test_list.txt',  SONY_DIR)
    print(f'Train: {len(train_pairs)} pairs')
    print(f'Val  : {len(val_pairs)}   pairs')
    print(f'Test : {len(test_pairs)}  pairs')
else:
    # Synthetic placeholders — the SIDDataset will handle these
    train_pairs = [{'input_path': None, 'gt_path': None, 'amp_ratio': 300.0}] * DEMO_N_TRAIN
    val_pairs   = [{'input_path': None, 'gt_path': None, 'amp_ratio': 300.0}] * DEMO_N_VAL
    test_pairs  = [{'input_path': None, 'gt_path': None, 'amp_ratio': 300.0}] * DEMO_N_TEST
    print('Synthetic demo pairs created.')

In [ ]:
# ────────────────────────────────────────────────────────────────
#  SIDDataset — PyTorch Dataset
# ────────────────────────────────────────────────────────────────

class SIDDataset(Dataset):
    """Sony subset of the See-in-the-Dark dataset.

    Training mode:
        Returns randomly cropped, augmented patches.
        Input  : (4, patch//2, patch//2)  — packed Bayer
        Target : (3, patch,    patch)     — sRGB GT

    Evaluation mode:
        Returns the full (scaled) image without cropping.
    """

    def __init__(self,
                 pairs: list,
                 patch_size: int = 512,
                 scale: float = 1.0,
                 is_train: bool = True,
                 cache_gt: bool = True):
        """
        pairs      — list of dicts from parse_sony_list()
        patch_size — crop size at original raw resolution
        scale      — downscale factor before cropping
        is_train   — if True, random crop + augment; else full image
        cache_gt   — cache GT images in RAM (speeds up training)
        """
        self.pairs      = pairs
        self.patch_size = patch_size
        self.scale      = scale
        self.is_train   = is_train
        self.gt_cache   = {}  # gt_path -> float32 numpy array
        self.do_cache   = cache_gt

    def __len__(self):
        return len(self.pairs)

    # ── internal helpers ────────────────────────────────────────

    def _get_gt(self, gt_path: str) -> np.ndarray:
        """Load (and optionally cache) the GT sRGB image."""
        if USE_SYNTHETIC_DEMO:
            H = max(4, int(DEMO_H * self.scale))
            W = max(4, int(DEMO_W * self.scale))
            return np.random.rand(H, W, 3).astype(np.float32)
        if self.do_cache and gt_path in self.gt_cache:
            return self.gt_cache[gt_path]
        gt = load_gt_rgb(gt_path, scale=self.scale)
        if self.do_cache:
            self.gt_cache[gt_path] = gt
        return gt

    def _get_input(self, in_path: str, amp: float) -> np.ndarray:
        """Load the packed + amplified input."""
        if USE_SYNTHETIC_DEMO:
            H = max(2, int(DEMO_H * self.scale)) // 2
            W = max(2, int(DEMO_W * self.scale)) // 2
            return np.clip(
                np.random.rand(H, W, 4).astype(np.float32) * amp, 0.0, 1.0
            )
        return load_input_raw(in_path, amp_ratio=amp, scale=self.scale)

    @staticmethod
    def _augment(inp: np.ndarray, gt: np.ndarray):
        """Apply random flip and 90° rotation to both input and GT."""
        # Random horizontal flip
        if random.random() > 0.5:
            inp = inp[:, ::-1, :].copy()
            gt  = gt[:, ::-1, :].copy()
        # Random vertical flip
        if random.random() > 0.5:
            inp = inp[::-1, :, :].copy()
            gt  = gt[::-1, :, :].copy()
        # Random 90° rotation (0 / 90 / 180 / 270)
        k = random.randint(0, 3)
        if k > 0:
            inp = np.rot90(inp, k, axes=(0, 1)).copy()
            gt  = np.rot90(gt,  k, axes=(0, 1)).copy()
        return inp, gt

    # ── Dataset interface ────────────────────────────────────────

    def __getitem__(self, idx):
        pair  = self.pairs[idx]
        amp   = float(pair['amp_ratio'])

        gt  = self._get_gt(pair['gt_path'])        # (H,  W,  3)  float32
        inp = self._get_input(pair['input_path'], amp)  # (H/2, W/2, 4)  float32

        if self.is_train:
            # ── Random crop ─────────────────────────────────────
            ps   = self.patch_size
            ps_h = ps // 2  # packed height patch

            H_in, W_in = inp.shape[:2]
            H_gt, W_gt = gt.shape[:2]

            # Ensure patch fits
            if H_in < ps_h or W_in < ps_h:
                ps_h = min(H_in, W_in)
                ps   = ps_h * 2

            max_y = H_in - ps_h
            max_x = W_in - ps_h
            if max_y <= 0 or max_x <= 0:
                y_in, x_in = 0, 0
            else:
                y_in = random.randint(0, max_y)
                x_in = random.randint(0, max_x)

            # Corresponding GT crop (2× spatial resolution)
            y_gt = y_in * 2
            x_gt = x_in * 2

            inp_patch = inp[y_in:y_in+ps_h, x_in:x_in+ps_h, :]
            gt_patch  = gt[y_gt:y_gt+ps,   x_gt:x_gt+ps,   :]

            inp_patch, gt_patch = self._augment(inp_patch, gt_patch)
        else:
            inp_patch = inp
            gt_patch  = gt

        # ── Numpy → Torch (C, H, W) ──────────────────────────────
        inp_t = torch.from_numpy(
            np.transpose(inp_patch, (2, 0, 1)).copy()
        )  # (4, H/2, W/2)
        gt_t  = torch.from_numpy(
            np.transpose(gt_patch,  (2, 0, 1)).copy()
        )  # (3, H, W)

        return inp_t, gt_t


# ── Create datasets & loaders ────────────────────────────────────

train_dataset = SIDDataset(
    train_pairs, patch_size=PATCH_SIZE, scale=SCALE_FACTOR,
    is_train=True, cache_gt=True
)
val_dataset   = SIDDataset(
    val_pairs, patch_size=PATCH_SIZE, scale=SCALE_FACTOR,
    is_train=False, cache_gt=True
)
test_dataset  = SIDDataset(
    test_pairs, patch_size=PATCH_SIZE, scale=SCALE_FACTOR,
    is_train=False, cache_gt=False
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)
val_loader = DataLoader(
    val_dataset, batch_size=1, shuffle=False,
    num_workers=0, pin_memory=False,
)

print(f'Train batches : {len(train_loader)}')
print(f'Val   samples : {len(val_dataset)}')
print(f'Test  samples : {len(test_dataset)}')

# Quick shape sanity check
x, y = next(iter(train_loader))
print(f'Input  batch : {tuple(x.shape)}  dtype={x.dtype}')
print(f'Target batch : {tuple(y.shape)}  dtype={y.dtype}')

---
## Section 6 — U-Net Model

The paper's default architecture is a **U-Net** [Ronneberger et al., 2015] with the following adaptations:

* **Input**: 4-channel packed Bayer at half spatial resolution.
* **Encoder**: 5 levels, each with 2 × (Conv → LeakyReLU), followed by MaxPool(2).
  Channel widths: 32 → 64 → 128 → 256 → 512.
* **Bottleneck**: 2 × (Conv → LeakyReLU) at 512 channels.
* **Decoder**: ConvTranspose upsampling + skip-connection concat + 2 × Conv per level.
* **Output head**: 1 × 1 Conv → 12 channels → `PixelShuffle(2)` → 3-channel sRGB at full resolution.
* **No batch normalisation** (hurts performance on varying amplification ratios).
* **No residual connections** (input/output are in different colour spaces).

In [ ]:
# ────────────────────────────────────────────────────────────────
#  U-Net building blocks
# ────────────────────────────────────────────────────────────────

def conv_block(in_ch: int, out_ch: int) -> nn.Sequential:
    """Two Conv-LeakyReLU layers (kernel 3×3, padding 1)."""
    return nn.Sequential(
        nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=True),
        nn.LeakyReLU(0.2, inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=True),
        nn.LeakyReLU(0.2, inplace=True),
    )


class SIDUNet(nn.Module):
    """U-Net for raw low-light image processing (Sony Bayer subset).

    Input  : (B, 4, H/2, W/2)   — packed amplified Bayer
    Output : (B, 3, H,   W)     — predicted sRGB, clamped to [0, 1]
    """

    def __init__(self, in_channels: int = 4, base_ch: int = 32):
        super().__init__()
        ch = base_ch  # 32

        # ── Encoder ─────────────────────────────────────────────
        self.enc1 = conv_block(in_channels, ch)       # 32
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = conv_block(ch,   ch*2)            # 64
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = conv_block(ch*2, ch*4)            # 128
        self.pool3 = nn.MaxPool2d(2)

        self.enc4 = conv_block(ch*4, ch*8)            # 256
        self.pool4 = nn.MaxPool2d(2)

        # ── Bottleneck ───────────────────────────────────────────
        self.bottleneck = conv_block(ch*8, ch*16)     # 512

        # ── Decoder ─────────────────────────────────────────────
        self.up4   = nn.ConvTranspose2d(ch*16, ch*8, 2, stride=2)
        self.dec4  = conv_block(ch*16, ch*8)          # concat → 512

        self.up3   = nn.ConvTranspose2d(ch*8, ch*4, 2, stride=2)
        self.dec3  = conv_block(ch*8,  ch*4)          # concat → 256

        self.up2   = nn.ConvTranspose2d(ch*4, ch*2, 2, stride=2)
        self.dec2  = conv_block(ch*4,  ch*2)          # concat → 128

        self.up1   = nn.ConvTranspose2d(ch*2, ch,   2, stride=2)
        self.dec1  = conv_block(ch*2,  ch)            # concat → 64

        # ── Output: 12 channels → PixelShuffle(2) → 3 channels ──
        self.out_conv    = nn.Conv2d(ch, 12, 1)       # 1×1 conv
        self.pixel_shuffle = nn.PixelShuffle(2)       # (B,12,H/2,W/2)→(B,3,H,W)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, a=0.2, nonlinearity='leaky_relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ── Encoder ─────────────────────────────────────────────
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        # ── Bottleneck ───────────────────────────────────────────
        b  = self.bottleneck(self.pool4(e4))

        # ── Decoder (with skip connections) ──────────────────────
        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        # ── Output ───────────────────────────────────────────────
        out = self.pixel_shuffle(self.out_conv(d1))  # (B, 3, H, W)
        return torch.clamp(out, 0.0, 1.0)


# ── Instantiate and inspect ──────────────────────────────────────
model = SIDUNet(in_channels=4, base_ch=32).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable:,}')

# Forward-pass smoke test
with torch.no_grad():
    dummy_in  = torch.rand(1, 4, PATCH_SIZE // 2, PATCH_SIZE // 2).to(device)
    dummy_out = model(dummy_in)
print(f'Smoke test — input : {tuple(dummy_in.shape)}')
print(f'Smoke test — output: {tuple(dummy_out.shape)}')
assert dummy_out.shape == (1, 3, PATCH_SIZE, PATCH_SIZE), \
    f'Unexpected output shape: {dummy_out.shape}'
print('Model architecture OK.')

---
## Section 7 — Training

### Training recipe (from paper)
| Detail | Value |
|--------|-------|
| Loss | L1 |
| Optimiser | Adam (β₁=0.9, β₂=0.999) |
| Initial LR | 1 × 10⁻⁴ |
| LR schedule | Decay to 1 × 10⁻⁵ after 2,000 epochs |
| Epochs | 4,000 |
| Crop size | 512 × 512 |
| Augmentation | Random flip + 90° rotation |

### GPU optimisations added
* **Automatic Mixed Precision (AMP)** — `torch.cuda.amp.autocast` + `GradScaler`.
* `pin_memory=True` in DataLoader for faster host→device transfers.
* Gradient clipping to stabilise training.

In [ ]:
# ────────────────────────────────────────────────────────────────
#  Training utilities
# ────────────────────────────────────────────────────────────────

criterion = nn.L1Loss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR_INITIAL,
    betas=(0.9, 0.999),
    eps=1e-8,
)

_amp_device = 'cuda' if torch.cuda.is_available() else 'cpu'
scaler = GradScaler(_amp_device, enabled=torch.cuda.is_available())


def set_lr(optimizer, lr: float):
    for g in optimizer.param_groups:
        g['lr'] = lr


def train_epoch(model, loader, optimizer, scaler, device) -> float:
    """Run one training epoch. Returns mean L1 loss."""
    model.train()
    total_loss = 0.0
    for inputs, targets in loader:
        inputs  = inputs.to(device,  non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(_amp_device, enabled=torch.cuda.is_available()):
            preds = model(inputs)
            loss  = criterion(preds, targets)

        scaler.scale(loss).backward()
        # Gradient clipping helps with numerical stability
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device) -> dict:
    """Evaluate on a loader. Returns dict with mean PSNR, SSIM, L1 loss."""
    model.eval()
    psnrs, ssims, losses = [], [], []

    for inputs, targets in loader:
        inputs  = inputs.to(device)
        targets = targets.to(device)

        with autocast(_amp_device, enabled=torch.cuda.is_available()):
            preds = model(inputs)
            loss  = criterion(preds, targets)

        losses.append(loss.item())

        # Move to CPU for metric computation
        pred_np = preds[0].cpu().float().numpy().transpose(1, 2, 0)    # H W C
        tgt_np  = targets[0].cpu().float().numpy().transpose(1, 2, 0)
        pred_np = np.clip(pred_np, 0.0, 1.0)
        tgt_np  = np.clip(tgt_np,  0.0, 1.0)

        p = psnr_fn(tgt_np, pred_np, data_range=1.0)
        s = ssim_fn(
            tgt_np, pred_np,
            data_range=1.0,
            channel_axis=2,
        _min_dim = min(tgt_np.shape[:2])
            win_size=min(7, _min_dim - (_min_dim % 2 == 0)),
        )
        psnrs.append(p)
        ssims.append(s)

    return {
        'loss': float(np.mean(losses)),
        'psnr': float(np.mean(psnrs)),
        'ssim': float(np.mean(ssims)),
    }


def save_checkpoint(model, optimizer, epoch: int, metrics: dict,
                    path: str):
    torch.save({
        'epoch':          epoch,
        'model_state':    model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'metrics':        metrics,
    }, path)


def load_checkpoint(model, optimizer, path: str) -> int:
    """Load checkpoint. Returns epoch number to resume from."""
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    print(f'Resumed from epoch {ckpt["epoch"]} '
          f'(PSNR={ckpt["metrics"].get("psnr", "?"):.2f})')
    return ckpt['epoch']


print('Training utilities ready.')

In [ ]:
# ── Optional: resume from checkpoint ────────────────────────────
START_EPOCH    = 0
BEST_PSNR      = 0.0
RESUME_CKPT    = None  # e.g. '/content/checkpoints/best_model.pth'

if RESUME_CKPT and Path(RESUME_CKPT).exists():
    START_EPOCH = load_checkpoint(model, optimizer, RESUME_CKPT)
else:
    print('Starting training from scratch.')

# History for plotting
history = {'train_loss': [], 'val_loss': [], 'val_psnr': [], 'val_ssim': []}

In [ ]:
# ════════════════════════════════════════════════════════════════
#  MAIN TRAINING LOOP
# ════════════════════════════════════════════════════════════════

print(f'Training for {NUM_EPOCHS} epochs  '
      f'(LR: {LR_INITIAL} → {LR_DECAYED} at epoch {LR_DECAY_EP})')
print('=' * 65)

t0 = time.time()

for epoch in range(START_EPOCH + 1, NUM_EPOCHS + 1):

    # ── Learning-rate schedule ───────────────────────────────────
    current_lr = LR_INITIAL if epoch <= LR_DECAY_EP else LR_DECAYED
    set_lr(optimizer, current_lr)

    # ── Train ────────────────────────────────────────────────────
    train_loss = train_epoch(model, train_loader, optimizer, scaler, device)
    history['train_loss'].append(train_loss)

    # ── Validate (every 50 epochs) ───────────────────────────────
    if epoch % 50 == 0 or epoch == 1:
        val_metrics = evaluate(model, val_loader, device)
        history['val_loss'].append(val_metrics['loss'])
        history['val_psnr'].append(val_metrics['psnr'])
        history['val_ssim'].append(val_metrics['ssim'])

        elapsed = time.time() - t0
        print(
            f'Ep {epoch:4d}/{NUM_EPOCHS} | '
            f'LR={current_lr:.0e} | '
            f'Train L1={train_loss:.4f} | '
            f'Val L1={val_metrics["loss"]:.4f} | '
            f'PSNR={val_metrics["psnr"]:.2f} dB | '
            f'SSIM={val_metrics["ssim"]:.4f} | '
            f't={elapsed/60:.1f}min'
        )

        # Save best model
        if val_metrics['psnr'] > BEST_PSNR:
            BEST_PSNR = val_metrics['psnr']
            save_checkpoint(
                model, optimizer, epoch, val_metrics,
                str(CKPT_DIR / 'best_model.pth')
            )
            print(f'  ↑ New best PSNR: {BEST_PSNR:.2f} dB — checkpoint saved.')

    # ── Periodic checkpoint ──────────────────────────────────────
    if epoch % SAVE_EVERY == 0:
        save_checkpoint(
            model, optimizer, epoch,
            {'psnr': BEST_PSNR},
            str(CKPT_DIR / f'epoch_{epoch:04d}.pth')
        )

print('\nTraining complete.')
print(f'Best validation PSNR: {BEST_PSNR:.2f} dB')

---
## Section 8 — Training Curves

In [ ]:
def plot_training_history(history: dict):
    epochs_train = list(range(1, len(history['train_loss']) + 1))
    # Val metrics are recorded every 50 epochs
    epochs_val   = list(range(50, len(history['val_psnr']) * 50 + 1, 50))
    if not epochs_val:
        epochs_val = [1]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(epochs_train, history['train_loss'], label='Train L1', color='steelblue')
    if history['val_loss']:
        axes[0].plot(epochs_val[:len(history['val_loss'])],
                     history['val_loss'],  label='Val L1',   color='tomato')
    axes[0].set_title('L1 Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('L1')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    if history['val_psnr']:
        axes[1].plot(epochs_val[:len(history['val_psnr'])],
                     history['val_psnr'], color='green')
    axes[1].set_title('Validation PSNR (dB)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('PSNR (dB)')
    axes[1].grid(True, alpha=0.3)

    if history['val_ssim']:
        axes[2].plot(epochs_val[:len(history['val_ssim'])],
                     history['val_ssim'], color='purple')
    axes[2].set_title('Validation SSIM')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('SSIM')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(RESULT_DIR / 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Curves saved to {RESULT_DIR / "training_curves.png"}')


if any(history.values()):
    plot_training_history(history)
else:
    print('No training history yet — run the training loop first.')

---
## Section 9 — Test-Set Evaluation

In [ ]:
# ── Load best checkpoint (if available) ──────────────────────────
best_ckpt = CKPT_DIR / 'best_model.pth'
if best_ckpt.exists():
    ckpt = torch.load(str(best_ckpt), map_location=device)
    model.load_state_dict(ckpt['model_state'])
    print(f'Loaded best model (epoch {ckpt["epoch"]}, '
          f'PSNR={ckpt["metrics"].get("psnr","?"):.2f} dB)')
else:
    print('No checkpoint found — using current model weights.')

# ── Run test evaluation ───────────────────────────────────────────
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

test_metrics = evaluate(model, test_loader, device)
print('\n── Test-set results ────────────────────────────────')
print(f'  L1 Loss : {test_metrics["loss"]:.4f}')
print(f'  PSNR    : {test_metrics["psnr"]:.2f} dB')
print(f'  SSIM    : {test_metrics["ssim"]:.4f}')
print('────────────────────────────────────────────────────')
print('\nPaper reports ~28.88 dB PSNR / 0.787 SSIM on Sony full-res.')
print('Lower values expected here due to scaled-down images and fewer epochs.')

---
## Section 10 — Visualisation

In [ ]:
# ────────────────────────────────────────────────────────────────
#  Visualise: (amplified raw | network output | GT) for N samples
# ────────────────────────────────────────────────────────────────

def tensor_to_image(t: torch.Tensor) -> np.ndarray:
    """Convert (C, H, W) float tensor [0,1] to (H, W, C) uint8."""
    img = t.cpu().float().numpy().transpose(1, 2, 0)
    img = np.clip(img, 0.0, 1.0)
    return (img * 255).astype(np.uint8)


def packed_to_display(packed: torch.Tensor) -> np.ndarray:
    """Convert packed 4-channel Bayer to a simple 3-channel RGB for display.
    Channels: R=0, Gr=1, B=2, Gb=3  → RGB using R, mean(Gr,Gb), B.
    """
    p = packed.cpu().float()  # (4, H, W)
    r  = p[0]
    g  = (p[1] + p[3]) / 2.0
    b  = p[2]
    rgb = torch.stack([r, g, b], dim=0)  # (3, H, W)
    return tensor_to_image(rgb)


@torch.no_grad()
def visualise_samples(model, dataset, n_samples: int = 4,
                      save_dir: Path = RESULT_DIR):
    model.eval()
    indices = random.sample(range(len(dataset)), min(n_samples, len(dataset)))

    fig, axes = plt.subplots(n_samples, 3,
                             figsize=(12, 4 * n_samples))
    if n_samples == 1:
        axes = axes[np.newaxis, :]

    col_titles = ['Input (amplified raw → RGB approx)',
                  'Network output', 'Ground truth']
    for j, title in enumerate(col_titles):
        axes[0, j].set_title(title, fontsize=11, fontweight='bold')

    for row, idx in enumerate(indices):
        inp_t, gt_t = dataset[idx]
        inp_t = inp_t.unsqueeze(0).to(device)

        with autocast(_amp_device, enabled=torch.cuda.is_available()):
            pred_t = model(inp_t)[0]

        inp_disp  = packed_to_display(inp_t[0])
        pred_disp = tensor_to_image(pred_t)
        gt_disp   = tensor_to_image(gt_t)

        # Compute per-sample metrics
        p = psnr_fn(
            gt_disp.astype(np.float32) / 255.0,
            pred_disp.astype(np.float32) / 255.0,
            data_range=1.0,
        )
        h, w = gt_disp.shape[:2]
        _min_hw = min(h, w)
        win = min(7, _min_hw - (_min_hw % 2 == 0))
        win = max(3, win)
        s = ssim_fn(
            gt_disp.astype(np.float32) / 255.0,
            pred_disp.astype(np.float32) / 255.0,
            data_range=1.0,
            channel_axis=2,
            win_size=win,
        )

        for j, img in enumerate([inp_disp, pred_disp, gt_disp]):
            axes[row, j].imshow(img)
            axes[row, j].axis('off')

        axes[row, 1].set_title(
            f'PSNR={p:.1f} dB  SSIM={s:.3f}',
            fontsize=9, color='darkgreen'
        )

        # Save individual outputs
        imageio.imwrite(str(save_dir / f'pred_{idx:04d}.png'), pred_disp)
        imageio.imwrite(str(save_dir / f'gt_{idx:04d}.png'),   gt_disp)

    plt.tight_layout()
    plt.savefig(str(save_dir / 'visual_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Results saved to {save_dir}')


visualise_samples(model, test_dataset, n_samples=min(4, len(test_dataset)))

---
## Section 11 — Inference on a Single Raw Image

Use this cell to run the trained model on any new `.ARW` (or supported raw) file.

In [ ]:
# ────────────────────────────────────────────────────────────────
#  Single-image inference
# ────────────────────────────────────────────────────────────────

@torch.no_grad()
def infer_single(
    raw_path: str,
    amp_ratio: float,
    model: nn.Module,
    device: torch.device,
    scale: float = SCALE_FACTOR,
    save_path: str | None = None,
) -> np.ndarray:
    """Run the pipeline on one raw image file.

    raw_path  — path to input ARW / CR2 / NEF file
    amp_ratio — amplification factor (e.g. 100, 250, 300)
    Returns predicted sRGB uint8 image (H, W, 3).
    """
    model.eval()

    if USE_SYNTHETIC_DEMO:
        H = max(2, int(DEMO_H * scale)) // 2
        W = max(2, int(DEMO_W * scale)) // 2
        packed = np.clip(
            np.random.rand(H, W, 4).astype(np.float32) * amp_ratio, 0.0, 1.0
        )
    else:
        packed = load_input_raw(raw_path, amp_ratio=amp_ratio, scale=scale)

    # (H, W, 4) → (1, 4, H, W)
    inp_t = torch.from_numpy(
        np.transpose(packed, (2, 0, 1))[np.newaxis]
    ).to(device)

    with autocast(_amp_device, enabled=torch.cuda.is_available()):
        pred = model(inp_t)[0]   # (3, H*2, W*2)

    out_np = pred.cpu().float().numpy().transpose(1, 2, 0)
    out_np = np.clip(out_np, 0.0, 1.0)
    out_u8 = (out_np * 255).astype(np.uint8)

    if save_path:
        imageio.imwrite(save_path, out_u8)
        print(f'Saved prediction to {save_path}')

    return out_u8


# ── Example usage ────────────────────────────────────────────────
# Replace the path and amp_ratio below with your own image.
EXAMPLE_RAW   = None   # e.g. '/content/SID/Sony/short/00001_00_0.1s.ARW'
EXAMPLE_AMP   = 300    # exposure ratio (long / short)

if EXAMPLE_RAW and Path(EXAMPLE_RAW).exists():
    pred_img = infer_single(
        EXAMPLE_RAW, EXAMPLE_AMP, model, device,
        save_path=str(RESULT_DIR / 'inference_output.png')
    )
    plt.figure(figsize=(8, 5))
    plt.imshow(pred_img)
    plt.title(f'Predicted sRGB  (amp × {EXAMPLE_AMP})')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
elif USE_SYNTHETIC_DEMO:
    pred_img = infer_single(None, 300, model, device,
                            save_path=str(RESULT_DIR / 'inference_demo.png'))
    plt.figure(figsize=(6, 4))
    plt.imshow(pred_img)
    plt.title('Synthetic demo inference output')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Set EXAMPLE_RAW to a valid .ARW path to run inference.')

---
## Section 12 — Paper Ablation Study Summary

The table below reproduces **Table 3** from the paper (Sony subset, PSNR dB / SSIM).
Use this as a reference when interpreting your training results.

| Condition | Sony | Fuji |
|-----------|------|------|
| **1. Default (U-Net, L1, raw input)** | **28.88 / 0.787** | **26.61 / 0.680** |
| 2. U-Net → CAN | 27.40 / 0.792 | 25.71 / 0.710 |
| 3. Raw → sRGB input | 17.40 / 0.554 | 25.11 / 0.648 |
| 4. L1 → SSIM loss | 28.64 / 0.817 | 26.20 / 0.685 |
| 5. L1 → L2 loss | 28.47 / 0.784 | 26.51 / 0.680 |
| 6. Packed → Masked Bayer | 26.95 / 0.744 | — |
| 8. Stretched references | 18.23 / 0.674 | 16.85 / 0.535 |

**Key findings from the paper:**
- Operating on **raw sensor data** (rather than processed sRGB) is critical (row 3 shows a 11+ dB drop).
- **Packing** the Bayer mosaic into 4 channels outperforms masking (row 6).
- **U-Net** outperforms the CAN architecture in PSNR (row 2).
- **L1 loss** is the paper's default; L2 and SSIM losses produce comparable results (rows 4–5).
- Do **not** apply histogram stretching to GT images during training (row 8).

In [ ]:
# ── Optional: SSIM-based loss (ablation experiment 4) ───────────
#
# Uncomment the block below to train with SSIM loss instead of L1.
# Requires the pytorch-msssim package.
#
# !pip install -q pytorch-msssim
# from pytorch_msssim import SSIM as SSIMLoss
#
# ssim_loss_fn = SSIMLoss(data_range=1.0, size_average=True, channel=3)
#
# def ssim_criterion(pred, target):
#     return 1.0 - ssim_loss_fn(pred, target)
#
# criterion = ssim_criterion   # replace L1 loss

print('Ablation cell loaded (no-op by default).')

---
## Summary

This notebook implements the full **Learning to See in the Dark** pipeline:

1. **Raw preprocessing** — 14-bit Bayer mosaic → 4-channel packed tensor, black-level subtraction, amplification by exposure ratio (100–300×).
2. **U-Net** — 5-level encoder-decoder with skip connections, LeakyReLU activations, no batch norm.
3. **PixelShuffle output** — 12-channel feature map → full-resolution 3-channel sRGB.
4. **Training** — L1 loss, Adam, step LR, AMP mixed-precision for GPU efficiency.
5. **Scaling** — `SCALE_FACTOR` and `PATCH_SIZE` allow you to trade image resolution for speed.

### Reproducibility notes
- The paper trains on **full-resolution images (4240 × 2832)** for 4,000 epochs. Reproducing exact PSNR numbers requires the full SID dataset and several GPU-days.
- With `SCALE_FACTOR=0.5` and `PATCH_SIZE=512`, training runs ~4× faster and produces results within ~1–2 dB of the full-resolution numbers.
- The synthetic demo (`USE_SYNTHETIC_DEMO=True`) lets you verify the entire pipeline end-to-end without any data.